### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="website_phishing",
    dataset_year="2014",
    domain_str="technology & internet",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5B301",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/website_phishing/ && wget -P local-data-warehouse/website_phishing/ https://archive.ics.uci.edu/static/public/379/website+phishing.zip && unzip local-data-warehouse/website_phishing/website+phishing.zip && rm local-data-warehouse/website_phishing/website+phishing.zip
""",
    # References
    academic_reference_bibtex="""@article{abdelhamid2014phishing,
  title={Phishing detection based associative classification data mining},
  author={Abdelhamid, Neda and Ayesh, Aladdin and Thabtah, Fadi},
  journal={Expert Systems with Applications},
  volume={41},
  number={13},
  pages={5948--5959},
  year={2014},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="abdelhamid2014phishing",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We reversed the ordinal encoding of the original data.
- We renamed the target feature to be more meaningful.
- Anomaly: the data has many duplicates (~50%).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="WebsiteType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="WebsiteType",
)

## Preprocessing

In [2]:
import arff
import pandas as pd

with open(f"{dataset_mold.path}/PhishingData.arff") as f:
    data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

# According to description this is the encoding for all features and the target
encoding_map = {"1": "Legitimate", "0": "Suspicious", "-1": "Phishy"}
df = df.map(lambda x: encoding_map[x])
target_feature = "WebsiteType"
df = df.rename(columns={"Result": target_feature})

cat_features = [
    "SFH",
    "popUpWidnow",
    "SSLfinal_State",
    "Request_URL",
    "URL_of_Anchor",
    "web_traffic",
    "URL_Length",
    "age_of_domain",
    "having_IP_Address",
    "WebsiteType",
]

df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,353
Columns: 10
Use sampling: False (sample size: 1,353)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['SFH', 'popUpWidnow', 'SSLfinal_State', 'Request_URL', 'URL_of_Anchor', 'web_traffic', 'URL_Length', 'age_of_domain', 'having_IP_Address']
Rows remaining as candidates after top-9 filter: 970 (of 1,353)

#### Duplicate Report
Total duplicate rows: 629 (46.49% of dataset)
Duplicate rows ignoring target: 677 (50.04% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,SFH,popUpWidnow,SSLfinal_State,Request_URL,URL_of_Anchor,web_traffic,URL_Length,age_of_domain,having_IP_Address,WebsiteType
0,Legitimate,Legitimate,Phishy,Phishy,Legitimate,Phishy,Legitimate,Legitimate,Suspicious,Suspicious
1,Phishy,Phishy,Suspicious,Phishy,Phishy,Legitimate,Phishy,Phishy,Suspicious,Legitimate
2,Legitimate,Suspicious,Legitimate,Phishy,Legitimate,Phishy,Suspicious,Legitimate,Suspicious,Phishy
3,Legitimate,Suspicious,Legitimate,Legitimate,Legitimate,Phishy,Suspicious,Legitimate,Suspicious,Phishy
4,Phishy,Phishy,Legitimate,Phishy,Phishy,Suspicious,Suspicious,Legitimate,Suspicious,Legitimate


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,SFH,category,0.0,0.0,3.0,"Legitimate, Phishy, Suspicious"
1,popUpWidnow,category,0.0,0.0,3.0,"Suspicious, Phishy, Legitimate"
2,SSLfinal_State,category,0.0,0.0,3.0,"Legitimate, Phishy, Suspicious"
3,Request_URL,category,0.0,0.0,3.0,"Phishy, Suspicious, Legitimate"
4,URL_of_Anchor,category,0.0,0.0,3.0,"Phishy, Legitimate, Suspicious"
5,web_traffic,category,0.0,0.0,3.0,"Suspicious, Legitimate, Phishy"
6,URL_Length,category,0.0,0.0,3.0,"Suspicious, Phishy, Legitimate"
7,age_of_domain,category,0.0,0.0,2.0,"Legitimate, Phishy"
8,having_IP_Address,category,0.0,0.0,2.0,"Suspicious, Legitimate"
9,WebsiteType,category,0.0,0.0,3.0,"Phishy, Legitimate, Suspicious"


In [6]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column            rank                          
Request_URL       1         Phishy    617  45.60
                  2     Suspicious    421  31.12
                  3     Legitimate    315  23.28
SFH               1     Legitimate    767  56.69
                  2         Phishy    445  32.89
                  3     Suspicious    141  10.42
SSLfinal_State    1     Legitimate    751  55.51
                  2         Phishy    308  22.76
                  3     Suspicious    294  21.73
URL_Length        1     Suspicious    563  41.61
                  2         Phishy    431  31.86
                  3     Legitimate    359  26.53
URL_of_Anchor     1         Phishy    610  45.08
                  2     Legitimate    576  42.57
                  3     Suspicious    167  12.34
WebsiteType       1         Phishy    702  51.88
                  2     Legitimate    548  40.50
                  3     Suspicious    103   7.61
age_of_domain     1     Legitimate    825  60.98
                  2         Phishy    528  39.02
having_IP_Address 1     Suspicious   1198  88.54
                  2     Legitimate    155  11.46
popUpWidnow       1     Suspicious    639  47.23
                  2         Phishy    532  39.32
                  3     Legitimate    182  13.45
web_traffic       1     Suspicious    473  34.96
                  2     Legitimate    440  32.52
                  3         Phishy    440  32.52

In [8]:
# Target Distribution
target_df

,count,pct
WebsiteType,,
Phishy,702,51.88
Legitimate,548,40.50
Suspicious,103,7.61


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to website_phishing/019d7369-f10c-78e6-bd19-b8b0a72399fe
019d7369-f10c-78e6-bd19-b8b0a72399fe
6104e2e749550eabee74aff56538d9c1bae06da15002fcfb52e836604d518e28
